# 01 · Data exploration
Event coverage, coordinate normalization and the 2024-25 scorekeeping break.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import polars as pl, numpy as np
import matplotlib.pyplot as plt
from src.config import load_config
cfg = load_config()
proc = cfg.path(cfg.data.processed_dir)
pl.Config.set_tbl_rows(25); pl.Config.set_tbl_width_chars(160)

In [ ]:
ev = pl.read_parquet(proc / 'events.parquet')
ev.group_by('season').agg(pl.col('game_id').n_unique().alias('games'), pl.len().alias('events')).sort('season')

In [ ]:
counts = ev.group_by('season', 'type').len().pivot(on='type', index='season', values='len').sort('season')
counts.select('season', 'giveaway', 'takeaway', 'hit', 'blocked-shot', 'missed-shot', 'shot-on-goal', 'goal')

Giveaways roughly double and takeaways fall from 2024-25: a recording change, handled as a separate **scoring regime**.

In [ ]:
sh = ev.filter(pl.col('type').is_in(['shot-on-goal','missed-shot','goal']))
fig, ax = plt.subplots(figsize=(8,3.5))
ax.hexbin(sh['x'], sh['y'], gridsize=60, cmap='Blues', mincnt=1)
ax.set_title('Unblocked shot locations after normalization (attacking +x)'); ax.set_aspect('equal')

In [ ]:
print((proc / 'data_quality.md').read_text())